In [ ]:
import os

### Write ```Dockerfile```

In [ ]:
%%writefile Dockerfile

FROM python:3.9
    
# update package manager
RUN apt-get update

# update pip
RUN pip install --upgrade pip

# copy requirements
COPY requirements.txt .

# install dependencies
RUN pip install -r requirements.txt

# copy script into container
COPY script.py .

# run script when image is run
CMD ["python3", "script.py"]

### Write ```requirements.txt``` to local drive

In [ ]:
%%writefile requirements.txt

pyarrow==19.0.1
fsspec==2025.3.2
s3fs==0.4.2

pandas==2.2.3
numpy==1.26.4

### Write ```script.py``` to local drive

In [ ]:
%%writefile script.py

import os
import pandas as pd

# get index of array
int_idx_array = int(os.environ['AWS_BATCH_JOB_ARRAY_INDEX'])
print(f'Array index: {int_idx_array}')

# constants
str_project = 'onboarding'
str_task = '07_aws_batch'
str_subtask = '02_parallel'

# make df
df = pd.DataFrame({
    'column': [int_idx_array],
})

# write to s3
str_filename = f'df_{int_idx_array}.csv'
str_uri = f's3://{str_project}/{str_task}/{str_subtask}/{str_filename}'
df.to_csv(str_uri, index=False)

### Build and push to ECR

In [ ]:
%%sh

# name the image
image=batch-example-parallel

# build image
docker build -t ${image} .

# get region
region=$(aws configure get region)
region=${region:-us-west-2}

# get account
account=$(aws sts get-caller-identity --query Account --output text)

# get full name
fullname="${account}.dkr.ecr.${region}.amazonaws.com/${image}:latest"

# get login command and execute it
aws ecr get-login-password --region "${region}" | docker login --username AWS --password-stdin "${account}".dkr.ecr."${region}".amazonaws.com

# create repository in ECR
aws ecr create-repository --repository-name "${image}" --image-scanning-configuration scanOnPush=true --image-tag-mutability MUTABLE

# tag image
docker tag  ${image} ${fullname}

# push image to ECR   
docker push ${fullname}

### Clean-up

In [ ]:
# rm files
for str_file in ['Dockerfile','requirements.txt','script.py']:
    try:
        os.remove(f'./{str_file}')
    except:
        pass